In [1]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub
import keras.backend as K
import matplotlib.pyplot as plt
from keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Permute, Reshape, LSTM, Dense, Dropout, Lambda
)
from glob import glob as gg
from pyspark.sql import SparkSession
from tensorflow.data import AUTOTUNE
from keras.callbacks import ModelCheckpoint
from keras.models import load_model, Model, Sequential
from pyspark.sql.functions import col, udf, aggregate as agg, max, min, filter
from pyspark.sql.types import StringType, ArrayType, IntegerType, FloatType

2025-04-25 13:07:50.982377: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-25 13:07:50.997769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-25 13:07:51.013479: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-25 13:07:51.018279: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-25 13:07:51.032275: I tensorflow/core/platform/cpu_feature_guar

In [2]:
spark = SparkSession.builder \
            .appName("LSTMJsonExtractor") \
            .config("spark.driver.port", "8887") \
            .config("spark.blockManager.port", "8886") \
            .config("spark.driver.memory",  "32g") \
            .config("spark.executor.memory","32g") \
            .config("spark.driver.maxResultSize", "32g") \
            .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/25 13:07:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/25 13:07:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# archive = kagglehub.dataset_download("aidapearson/ocr-data")# Cluster

archive = "/home2/va23abb/.cache/kagglehub/datasets/aidapearson/ocr-data/versions/36"

In [4]:
# gg(f"{archive}/*")

# archive

df = spark.read.parquet("images_small_dims.parquet")

df.show(5)

+--------------------+--------------------+--------------------+--------------------+--------------------+-----+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|            filename|               latex|    full_latex_chars| visible_latex_chars|    visible_char_map|width|height|depth|               xmins|               xmaxs|               ymins|               ymaxs|           xmins_raw|           xmaxs_raw|           ymins_raw|           ymaxs_raw|     true_file_paths|           png_masks|
+--------------------+--------------------+--------------------+--------------------+--------------------+-----+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------

In [5]:
df2 = df.select("true_file_paths", "visible_char_map")

In [6]:
with open(f"{archive}/extras/visible_char_map.json", "r") as file:
    symbols_index = json.load(file)

In [7]:
total_len = df2.count()

train_len = int(0.5 * total_len)
test_len = int(train_len * 0.5)
val_len = total_len - train_len - test_len

assert train_len + test_len + val_len == total_len

In [8]:
def generator():
    for row in df2.select("true_file_paths", "visible_char_map") \
        .toLocalIterator():
        path = row["true_file_paths"]
        seq = row["visible_char_map"]
        seq = np.array(seq, dtype = np.int32)
        if path is None:
            continue
        yield path, seq

In [9]:
def _load(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [32, 128])
    img = img / 255.0
    label = tf.cast(label, tf.int32)
    return img, label

In [10]:
def prepare(dataset, batch_size = 8):
	return (
		dataset
		.map(_load, num_parallel_calls = AUTOTUNE)
		.padded_batch(
			batch_size,
			padded_shapes = ([64, 256, 3], [64]),
			padding_values = (0.0, 0)
		)
		.prefetch(AUTOTUNE)
	)

In [11]:
raw_ds = tf.data.Dataset.from_generator(
	generator,
    output_signature = (
        tf.TensorSpec(shape = (), dtype = tf.string),
        tf.TensorSpec(shape = (None, ), dtype = tf.int32)
	)
)

2025-04-25 13:08:11.325998: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 858 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:17:00.0, compute capability: 8.0
2025-04-25 13:08:11.328108: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 78769 MB memory:  -> device: 1, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:65:00.0, compute capability: 8.0
2025-04-25 13:08:11.330168: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 78769 MB memory:  -> device: 2, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:ca:00.0, compute capability: 8.0
2025-04-25 13:08:11.331831: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 78769 MB memory:  -> device: 3, name: NVIDIA A100 80GB PCIe, pci bus 

In [12]:
raw_ds = raw_ds.shuffle(buffer_size = total_len, seed = 42)

In [13]:
train_ds = raw_ds.take(train_len)

rest_ds = raw_ds.skip(train_len)

test_ds = rest_ds.skip(test_len)

val_ds = rest_ds.take(test_len)

In [14]:
train_ds = prepare(train_ds, batch_size = 8)
test_ds = prepare(test_ds, batch_size = 8)
val_ds = prepare(val_ds, batch_size = 8)

In [17]:
len(symbols_index)

91

In [ ]:
img_h, img_w, n_ch = 64, 256, 3
n_classes = 91
max_timesteps = 64

input = Input(shape=(img_h, img_w, n_ch), name = "image_input")

x = Conv2D(32, 3, activation = "relu", padding = "same")(input)
x = BatchNormalization()(x)
x = MaxPooling2D(2)(x)

x = Conv2D(64, 3, activation="relu", padding = "same")(x)
x = BatchNormalization()(x)
x = MaxPooling2D(2)(x)

x = Permute((2,1,3))(x)
time_steps = x.shape[1]
features   = x.shape[2] * x.shape[3]
x = Reshape((time_steps, features))(x)

x = LSTM(128, return_sequences = True)(x)

output = Dense(n_classes, activation = "softmax")(x)

model = Model(inputs = input, outputs = output)

In [19]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

In [20]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 64, 256, 3)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 256, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 256, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 128, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 128, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 128, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 64, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 64, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64, 128)        │       590,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64, 91)         │        11,739 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 621,851 (2.37 MB)

 Trainable params: 621,659 (2.37 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
best_model_cb = ModelCheckpoint(
    filepath = "models/lstm_model.keras",
    monitor = "val_accuracy",
    save_best_only = True,
    mode = "max",
	verbose = 1
)

In [23]:
lstm_model = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = 10,
    callbacks = [best_model_cb]
)

Epoch 1/10


2025-04-25 13:09:37.526462: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90101
W0000 00:00:1745582977.722777 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.744842 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.745231 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.745554 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.745860 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.746207 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.746620 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582977.746968 2151364 gpu_

      4/Unknown 7s 20ms/step - accuracy: 0.4106 - loss: 3.1294

W0000 00:00:1745582978.795687 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.796134 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.796450 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.796752 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.797057 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.797363 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.797673 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.797982 2151364 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1745582978.798293 2151364 gp

    780/Unknown 21s 18ms/step - accuracy: 0.8485 - loss: 0.7530

...

2025-04-25 13:15:13.248539: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 16389235465421689108
2025-04-25 13:15:13.248655: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 15082343981458062589
2025-04-25 13:15:13.248682: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 9631959957145543567


780/780 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8747 - loss: 0.5269

2025-04-25 13:15:37.707170: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 11936654639572206366
2025-04-25 13:15:37.707301: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 9689236419140125957
2025-04-25 13:15:37.707330: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 110280975064374893
2025-04-25 13:15:37.707360: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 3585626214099651985



Epoch 10: val_accuracy did not improve from 0.87394
780/780 ━━━━━━━━━━━━━━━━━━━━ 38s 36ms/step - accuracy: 0.8747 - loss: 0.5269 - val_accuracy: 0.8727 - val_loss: 0.5437


2025-04-25 13:15:50.847658: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 16389235465421689108
2025-04-25 13:15:50.848680: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 15082343981458062589
2025-04-25 13:15:50.848790: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 9631959957145543567


In [24]:
model_history = model.history

In [29]:
with open('models/seg_training_history_02.json', 'w') as file:
    json.dump(lstm_model.history, file)

NameError: name 'lstm_model' is not defined

In [28]:
train_accuracy = model_history["accuracy"]
val_accuracy = model_history["val_accuracy"]
train_loss = model_history["loss"]
val_loss = model_history["val_loss"]

TypeError: 'History' object is not subscriptable